<a href="https://colab.research.google.com/github/samizard2016/data_analytics/blob/main/Fixed_effect_and_random_effect_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Fixed Effects vs Random Effects

Both models are used with panel data (data over time for the same entities — like people, firms, countries).

Example dataset idea:
person_id	year	income	education	experience
🔒 Fixed Effects Model (FE)

Key idea:
Control for all time-invariant differences between individuals.

👉 Each entity gets its own intercept.

Intuition
People differ in unobserved ways (ability, personality, etc.)
These don’t change over time
FE removes those effects
Model form:
𝑌
𝑖
𝑡
=
𝛽
𝑋
𝑖
𝑡
+
𝛼
𝑖
+
𝜖
𝑖
𝑡
Y
it
	​

=βX
it
	​

+α
i
	​

+ϵ
it
	​

𝛼
𝑖
α
i
	​

: individual-specific effect (fixed)
We allow correlation between
𝛼
𝑖
α
i
	​

 and
𝑋
𝑖
𝑡
X
it
	​

When to use FE:
You think unobserved individual traits affect your predictors
Example:
Ability affects both education and income → use FE
🎲 Random Effects Model (RE)

Key idea:
Individual differences are random and uncorrelated with predictors.

👉 One common intercept + random variation.

Model form:
𝑌
𝑖
𝑡
=
𝛽
𝑋
𝑖
𝑡
+
𝑢
𝑖
+
𝜖
𝑖
𝑡
Y
it
	​

=βX
it
	​

+u
i
	​

+ϵ
it
	​

𝑢
𝑖
u
i
	​

: random effect

Assumes:

𝐶
𝑜
𝑣
(
𝑋
𝑖
𝑡
,
𝑢
𝑖
)
=
0
Cov(X
it
	​

,u
i
	​

)=0
When to use RE:
Individual-specific effects are not correlated with regressors
Example:
Random variation across firms unrelated to inputs
⚖️ FE vs RE Summary
Feature	Fixed Effects	Random Effects
Individual intercepts	Yes (separate)	No (random)
Correlation with X	Allowed	Not allowed
Efficiency	Less efficient	More efficient
Bias risk	Low	High if assumption violated

In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_individuals = 100
n_years = 5

ids = np.repeat(np.arange(n_individuals), n_years)
years = np.tile(np.arange(2000, 2005), n_individuals)

# Variables
education = np.random.normal(12, 2, len(ids))
experience = np.random.normal(10, 3, len(ids))

# Individual fixed effect (unobserved)
alpha = np.repeat(np.random.normal(0, 5, n_individuals), n_years)

# Outcome
income = 2 * education + 1.5 * experience + alpha + np.random.normal(0, 2, len(ids))

df = pd.DataFrame({
    "id": ids,
    "year": years,
    "income": income,
    "education": education,
    "experience": experience
})
df["education"] = df["education"].round().astype(int)

df = df.set_index(["id", "year"])

In [4]:
df.head()

income  education  experience
id year                                  
0  2000  54.147453         13   12.778533
   2001  48.243584         12   15.728250
   2002  46.470727         13    5.804297
   2003  54.343079         15   11.688908
   2004  44.348638         12    8.048072

In [5]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 8.1 MB/s eta 0:00:00


In [6]:
from linearmodels.panel import PanelOLS

fe_model = PanelOLS.from_formula(
    "income ~ education + experience + EntityEffects",
    data=df
)

fe_results = fe_model.fit()
print(fe_results)

                          PanelOLS Estimation Summary                           
Dep. Variable:                 income   R-squared:                        0.8875
Estimator:                   PanelOLS   R-squared (Between):              0.9850
No. Observations:                 500   R-squared (Within):               0.8875
Date:                Thu, Mar 26 2026   R-squared (Overall):              0.9832
Time:                        23:31:54   Log-likelihood                   -1012.3
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      1569.8
Entities:                         100   P-value                           0.0000
Avg Obs:                       5.0000   Distribution:                   F(2,398)
Min Obs:                       5.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             1569.8
                            

In [7]:
from linearmodels.panel import RandomEffects

re_model = RandomEffects.from_formula(
    "income ~ education + experience",
    data=df
)

re_results = re_model.fit()
print(re_results)

                        RandomEffects Estimation Summary                        
Dep. Variable:                 income   R-squared:                        0.9508
Estimator:              RandomEffects   R-squared (Between):              0.9850
No. Observations:                 500   R-squared (Within):               0.8875
Date:                Fri, Mar 27 2026   R-squared (Overall):              0.9833
Time:                        00:21:24   Log-likelihood                   -1067.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      4813.9
Entities:                         100   P-value                           0.0000
Avg Obs:                       5.0000   Distribution:                   F(2,498)
Min Obs:                       5.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             4813.9
                            

Step 4: Hausman Test (Which to Choose?)

Used to decide FE vs RE.

Interpretation:
If coefficients differ significantly → use FE
If similar → RE is OK (more efficient)

In [8]:
from linearmodels.panel import compare

print(compare({"FE": fe_results, "RE": re_results}))

                    Model Comparison                    
                                    FE                RE
--------------------------------------------------------
Dep. Variable                   income            income
Estimator                     PanelOLS     RandomEffects
No. Observations                   500               500
Cov. Est.                   Unadjusted        Unadjusted
R-squared                       0.8875            0.9508
R-Squared (Within)              0.8875            0.8875
R-Squared (Between)             0.9850            0.9850
R-Squared (Overall)             0.9832            0.9833
F-statistic                     1569.8            4813.9
P-value (F-stat)                0.0000            0.0000
=====================     ============   ===============
education                       2.0243            2.0354
                              (38.783)          (57.117)
experience                      1.5391            1.5410
                              (

🧩 Final Decision Logic
✔ Choose RE if:
Coefficients ≈ FE (like yours)
Theory says no correlation
You want efficiency
✔ Choose FE if:
You suspect omitted variable bias
You want robustness
This is observational real-world data
🧠 Practical Rule (what economists actually do)

“Use FE unless you have a strong reason not to.”

Because:

RE can be biased
FE is safe
🧪 If you want to be rigorous

Run a Hausman test (not shown in your output):

If p-value < 0.05 → FE
If p-value > 0.05 → RE
✅ Your Case (Final Answer)

From your results:

👉 Statistically:

FE ≈ RE → RE is fine

👉 Practically:

If this were real data → I would still lean FE

👉 For your simulation:

RE is the correct choice

Hausman test step by step

In [9]:
import numpy as np

b_FE = fe_results.params[["education", "experience"]].values
b_RE = re_results.params[["education", "experience"]].values

In [10]:
b_diff = b_FE - b_RE

In [11]:
V_FE = fe_results.cov.loc[["education", "experience"], ["education", "experience"]].values
V_RE = re_results.cov.loc[["education", "experience"], ["education", "experience"]].values

In [12]:
V_diff = V_FE - V_RE

In [13]:
from numpy.linalg import inv

H = b_diff.T @ inv(V_diff) @ b_diff

| p-value | Decision                  |
| ------- | ------------------------- |
| < 0.05  | Reject H₀ → use FE        |
| ≥ 0.05  | Fail to reject → RE is OK |


In [14]:
from scipy.stats import chi2

df = len(b_diff)  # number of coefficients
p_value = 1 - chi2.cdf(H, df)

print("Hausman statistic:", H)
print("p-value:", p_value)

Hausman statistic: 0.420829811388044
p-value: 0.8102479997111169
